In [ ]:
# imports
# Requires: pip install torch torchvision scipy
import math
import time
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset, ConcatDataset
from torchvision import datasets, transforms
from torchvision.utils import save_image

In [ ]:
# 1. Hyperparameters & Noise Schedule
device = "cuda" if torch.cuda.is_available() else "cpu"
img_size = 32
T = 200                # More timesteps for finer RGB color transitions
batch_size = 32
epochs = 25
lr = 1e-3
num_images = 2560

In [ ]:
betas = torch.linspace(1e-4, 0.02, T, device=device)  # linear beta_t
alphas = 1.0 - betas                                  # alpha_t
alpha_hat = torch.cumprod(alphas, dim=0)              # alpha-bar_t

In [ ]:
# 2. Download & Pre-Cache Resized Flowers in RAM Once (~10 seconds, uses ~30 MB RAM)
# resize to 32x32, scale to [-1, 1]
transform = transforms.Compose([
    transforms.Resize((img_size, img_size)),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

# all three splits combined
full_ds = ConcatDataset([
    datasets.Flowers102(root="./data", split=s, download=True, transform=transform)
    for s in ("train", "val", "test")
])

print("Caching resized images in RAM...")
# fixed random subset of 2560 images, cached as one tensor
gen = torch.Generator().manual_seed(42)
indices = torch.randperm(len(full_ds), generator=gen)[:num_images].tolist()
cached_imgs = torch.stack([full_ds[i][0] for i in indices])
loader = DataLoader(TensorDataset(cached_imgs), batch_size=batch_size, shuffle=True, drop_last=True)

Caching resized images in RAM...


In [ ]:
# 3. Compact RGB U-Net (32 -> 64 -> 128 channels)
# sin/cos embedding of timestep t
class SinusoidalTimeEmb(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.dim = dim

    def forward(self, t):
        half = self.dim // 2
        freqs = torch.exp(-math.log(10000) * torch.arange(half, device=t.device) / (half - 1))
        args = t[:, None].float() * freqs[None, :]
        return torch.cat([torch.sin(args), torch.cos(args)], dim=-1)

# 2x (conv + GroupNorm + SiLU), time embedding added in between
class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch, time_dim):
        super().__init__()
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, padding=1)
        self.norm1 = nn.GroupNorm(8, out_ch)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, padding=1)
        self.norm2 = nn.GroupNorm(8, out_ch)
        self.time_mlp = nn.Linear(time_dim, out_ch)
        self.act = nn.SiLU()

    def forward(self, x, t_emb):
        h = self.act(self.norm1(self.conv1(x)))
        h = h + self.time_mlp(t_emb)[:, :, None, None]
        return self.act(self.norm2(self.conv2(h)))

# U-Net: predicts the noise from (x_t, t)
class FastFlowerUNet(nn.Module):
    def __init__(self, in_ch=3, time_dim=64):
        super().__init__()
        self.time_mlp = nn.Sequential(
            SinusoidalTimeEmb(time_dim),
            nn.Linear(time_dim, time_dim),
            nn.SiLU()
        )
        self.down1 = ConvBlock(in_ch, 32, time_dim)
        self.down2 = ConvBlock(32, 64, time_dim)
        self.pool = nn.MaxPool2d(2)

        self.bot = ConvBlock(64, 128, time_dim)

        self.up1 = nn.ConvTranspose2d(128, 64, 2, stride=2)
        self.dec1 = ConvBlock(128, 64, time_dim)
        self.up2 = nn.ConvTranspose2d(64, 32, 2, stride=2)
        self.dec2 = ConvBlock(64, 32, time_dim)
        self.out = nn.Conv2d(32, in_ch, 1)

    def forward(self, x, t):
        t_emb = self.time_mlp(t)
        # encoder + bottleneck
        d1 = self.down1(x, t_emb)
        d2 = self.down2(self.pool(d1), t_emb)
        b = self.bot(self.pool(d2), t_emb)

        # decoder with skip connections
        u1 = self.dec1(torch.cat([self.up1(b), d2], dim=1), t_emb)
        u2 = self.dec2(torch.cat([self.up2(u1), d1], dim=1), t_emb)
        return self.out(u2)

In [ ]:
# 4. Training Loop with Cosine LR Decay
model = FastFlowerUNet().to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
loss_fn = nn.MSELoss()

start_time = time.time()
for epoch in range(1, epochs + 1):
    total_loss = 0.0
    for (x0,) in loader:
        x0 = x0.to(device)
        # Instant in-memory horizontal flip augmentation
        if torch.rand(1).item() < 0.5:
            x0 = torch.flip(x0, dims=[3])

        t = torch.randint(0, T, (x0.size(0),), device=device)  # random t per image
        noise = torch.randn_like(x0)  # epsilon ~ N(0, I)

        a_hat = alpha_hat[t][:, None, None, None]
        xt = torch.sqrt(a_hat) * x0 + torch.sqrt(1.0 - a_hat) * noise  # forward process

        loss = loss_fn(model(xt, t), noise)  # MSE on predicted noise
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    scheduler.step()  # cosine LR decay
    elapsed = time.time() - start_time
    print(f"Epoch {epoch:02d}/{epochs} | Loss: {total_loss / len(loader):.4f} | Elapsed: {elapsed:.1f}s")

Epoch 01/25 | Loss: 0.3363 | Elapsed: 43.4s
Epoch 02/25 | Loss: 0.1595 | Elapsed: 85.5s
Epoch 03/25 | Loss: 0.1451 | Elapsed: 127.4s
Epoch 04/25 | Loss: 0.1411 | Elapsed: 169.5s
Epoch 05/25 | Loss: 0.1343 | Elapsed: 211.7s
Epoch 06/25 | Loss: 0.1312 | Elapsed: 254.1s
Epoch 07/25 | Loss: 0.1268 | Elapsed: 296.5s
Epoch 08/25 | Loss: 0.1256 | Elapsed: 339.1s
Epoch 09/25 | Loss: 0.1250 | Elapsed: 381.7s
Epoch 10/25 | Loss: 0.1203 | Elapsed: 423.7s
Epoch 11/25 | Loss: 0.1166 | Elapsed: 465.8s
Epoch 12/25 | Loss: 0.1237 | Elapsed: 507.6s
Epoch 13/25 | Loss: 0.1204 | Elapsed: 549.8s
Epoch 14/25 | Loss: 0.1161 | Elapsed: 592.0s
Epoch 15/25 | Loss: 0.1123 | Elapsed: 633.8s
Epoch 16/25 | Loss: 0.1175 | Elapsed: 675.6s
Epoch 17/25 | Loss: 0.1156 | Elapsed: 717.2s
Epoch 18/25 | Loss: 0.1100 | Elapsed: 758.8s
Epoch 19/25 | Loss: 0.1107 | Elapsed: 800.9s
Epoch 20/25 | Loss: 0.1127 | Elapsed: 842.7s
Epoch 21/25 | Loss: 0.1134 | Elapsed: 884.7s
Epoch 22/25 | Loss: 0.1105 | Elapsed: 926.5s
Epoch 23/25 

In [ ]:
# 5. Generate Final 4x4 Flower Grid (~10 seconds on CPU)
@torch.no_grad()
def sample(n_images=16):
    model.eval()
    x = torch.randn(n_images, 3, img_size, img_size, device=device)  # start from noise
    for t in reversed(range(T)):  # t = T-1, ..., 0
        t_batch = torch.full((n_images,), t, device=device, dtype=torch.long)
        pred_noise = model(x, t_batch)

        alpha_t = alphas[t]
        alpha_hat_t = alpha_hat[t]
        beta_t = betas[t]
        z = torch.randn_like(x) if t > 0 else 0  # no noise at last step

        # reverse step x_t -> x_{t-1}
        x = (1 / torch.sqrt(alpha_t)) * (
            x - ((1 - alpha_t) / torch.sqrt(1 - alpha_hat_t)) * pred_noise
        ) + torch.sqrt(beta_t) * z

    # back to [0, 1], save 4x4 grid
    x = (x.clamp(-1, 1) + 1) / 2
    save_image(x, "flowers_fast.png", nrow=4)
    print(f"Saved flowers_fast.png in {time.time() - start_time:.1f}s total!")

sample()

Saved flowers_fast.png in 1075.0s total!
